# ScholarLens 01 - Summarization (BART vs T5)
Owner: Member 1. Baselines -> fine-tune both models -> inspect predictions. Watch the progress-bar ETA in the first epoch; if a run won't finish inside your Colab session, lower `summarization.train_samples` in `configs/config.yaml` (and write the value you used in the report).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sajithaucm/Scholarlens_Project/blob/main/notebooks/01_summarization.ipynb)

## 0. Setup (run every time you open this notebook)
1. **Runtime -> Change runtime type -> T4 GPU** (Hardware accelerator).
2. The repo URL below already points to the team repo (sajithaucm/Scholarlens_Project).
3. Checkpoints are saved to **Google Drive** so they survive a disconnect; small result files are saved in the repo's `results/` folder and pushed to GitHub at the end.

In [ ]:
import os, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "https://github.com/sajithaucm/Scholarlens_Project.git"
if not os.path.exists("/content/Scholarlens_Project"):
    !git clone $REPO /content/Scholarlens_Project
%cd /content/Scholarlens_Project
!git pull -q
!pip install -q -r requirements.txt

# Checkpoints -> Google Drive (big files never go to GitHub)
os.environ["SCHOLARLENS_OUTPUT_DIR"] = "/content/drive/MyDrive/Scholarlens_Project/outputs"
os.makedirs(os.environ["SCHOLARLENS_OUTPUT_DIR"], exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip freeze > results/environment.txt   # exact library versions, for reproducibility

## 1. Quick test (2 minutes) - make sure everything runs on the GPU

In [ ]:
!python -m src.summarization.train --model t5-small --max-samples 64 --epochs 1 --no-bertscore

## 2. Baselines: Lead-3 and TextRank

In [ ]:
!python -m src.baselines --task summarization

## 3. Fine-tune BART-base (full run)

In [ ]:
!python -m src.summarization.train --model facebook/bart-base

## 4. Fine-tune T5-small (full run)

In [ ]:
!python -m src.summarization.train --model t5-small

## 5. Look at predictions (for error analysis: hallucinated facts, repetition, missing key results)

In [ ]:
import json
for line in list(open('results/summarization_facebook__bart-base_samples.jsonl'))[:3]:
    r = json.loads(line)
    print('REFERENCE:', r['reference'][:600], '\n\nBART:', r['prediction'][:600], '\n', '-' * 80)

In [ ]:
!python -m src.compare

## Save results to GitHub
Results (`results/*.json`, figures, sample predictions) are small and must be committed.
Create a GitHub **fine-grained personal access token** (repo contents: read & write) and paste it when asked. It is not stored in the notebook.
Alternative: download the `results/` folder (Files panel) and commit it from VS Code.

In [ ]:
%cd /content/Scholarlens_Project
from getpass import getpass
name, email = input("Your name: "), input("Your GitHub email: ")
token = getpass("GitHub token: ")
!git config user.name "$name" && git config user.email "$email"
!git pull -q
!git add results/
!git commit -m "Add results from Colab run" || echo "nothing to commit"
!git push https://$token@github.com/sajithaucm/Scholarlens_Project.git HEAD:main
del token